In [9]:
import requests

urls = [
    "https://api.listenbrainz.org/1/stats/sitewide/artists?range=all_time&count=5",
    "https://musicbrainz.org/ws/2/artist/5b11f4ce-a62d-471e-81fc-a69a8278c7da?fmt=json",
]

for u in urls:
    try:
        r = requests.get(
            u,
            timeout=20,
            headers={"User-Agent": "BreakoutArtistCapstone/1.0 (gabrielle.nelson16@gmail.com)"}
        )
        print("OK", r.status_code, u)
    except Exception as e:
        print("FAIL", type(e).__name__, u, str(e)[:140])

OK 200 https://api.listenbrainz.org/1/stats/sitewide/artists?range=all_time&count=5
OK 200 https://musicbrainz.org/ws/2/artist/5b11f4ce-a62d-471e-81fc-a69a8278c7da?fmt=json


In [10]:
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

LB_URL = "https://api.listenbrainz.org/1"
MB_URL = "https://musicbrainz.org/ws/2"
HEADERS = {"User-Agent": "BreakoutArtistCapstone/1.0 (gabrielle.nelson16@gmail.com)"}


def make_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    retry = Retry(
        total=6,
        connect=6,
        read=6,
        backoff_factor=1.5,  # waits 1.5s, 3s, 6s, ...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


SESSION = make_session()


def safe_get(url, params, attempts=5):
    """GET with retries that also catch SSL/connection errors."""
    for i in range(attempts):
        try:
            r = SESSION.get(url, params=params, timeout=30)
            return r
        except (requests.exceptions.SSLError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            wait = 2 ** i
            print(f"Request failed ({type(e).__name__}), retrying in {wait}s...")
            time.sleep(wait)
            # Fresh session in case the connection is stale
            globals()["SESSION"] = make_session()
    raise RuntimeError(f"Failed after {attempts} attempts: {url}")


def get_sitewide_top_artists(total=1000, time_range="all_time"):
    """Top artists across all of ListenBrainz, paged 100 at a time."""
    rows, offset = [], 0
    while len(rows) < total:
        r = safe_get(
            f"{LB_URL}/stats/sitewide/artists",
            {"range": time_range, "count": 100, "offset": offset},
        )
        if r.status_code == 204:
            break
        r.raise_for_status()

        payload = r.json().get("payload", {})
        artists = payload.get("artists", [])
        if not artists:
            break

        rows.extend(artists)
        offset += len(artists)
        time.sleep(0.5)

    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["listen_rank", "artist_name", "artist_mbid", "listen_count"])

    # robust to either artist_mbids (list) or artist_mbid (string)
    def extract_mbid(row):
        mbids = row.get("artist_mbids")
        if isinstance(mbids, list) and len(mbids) > 0:
            return mbids[0]
        mbid = row.get("artist_mbid")
        if isinstance(mbid, str) and mbid.strip():
            return mbid.strip()
        return None

    df["artist_mbid"] = df.apply(extract_mbid, axis=1)

    # guard missing columns
    if "artist_name" not in df.columns:
        df["artist_name"] = None
    if "listen_count" not in df.columns:
        df["listen_count"] = None

    df = df.dropna(subset=["artist_mbid"]).drop_duplicates(subset=["artist_mbid"]).reset_index(drop=True)
    df["listen_rank"] = df.index + 1
    return df[["listen_rank", "artist_name", "artist_mbid", "listen_count"]]

In [11]:
top_us_df = get_top_us_artists(n=50, candidate_pool=1000, time_range="all_time")
top_us_df.head()

,listen_rank,artist_name,artist_mbid,listen_count,country,area,type,begin_year
0,2,Linkin Park,f59c5520-5f46-4d2c-b2c4-822eabf53419,3115206,US,United States,Group,1996
1,5,Kendrick Lamar,381086ea-f511-4aba-bdf9-71c753dc5077,2882548,US,United States,Person,1987
2,6,Ye,164f0d73-1234-4e2c-8743-d77bf2191051,2861050,US,United States,Person,1977
3,9,Eminem,b95ce3ff-3d05-4e87-9e01-c97b66af13d4,2430743,US,United States,Person,1972
4,12,Nirvana,5b11f4ce-a62d-471e-81fc-a69a8278c7da,2323956,US,United States,Group,1987
